# Practice 2 — Prompt Engineering & Structured Output

Learn to control the LLM with better prompts and typed outputs.

**Topics:**
- Prompt templates
- System / human / AI messages
- Role prompting
- Few-shot examples
- Output formatting
- Pydantic structured output

In [ ]:
from dotenv import load_dotenv
import os
import warnings
warnings.filterwarnings("ignore")

load_dotenv(override=True)
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY", "")
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY", "")

from langchain_openai import ChatOpenAI
from langchain_groq import ChatGroq

if os.environ.get("OPENAI_API_KEY"):
    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.5)
    print("Using OpenAI")
elif os.environ.get("GROQ_API_KEY"):
    llm = ChatGroq(model="llama3-8b-8192", temperature=0.5)
    print("Using Groq")
else:
    raise ValueError("Set OPENAI_API_KEY or GROQ_API_KEY in .env")

## 1 — Basic Prompt Template

A prompt template lets you reuse the same structure with different inputs.

In [ ]:
from langchain_core.prompts import PromptTemplate

template = PromptTemplate.from_template(
    "Explain {topic} to a {audience} in {sentences} sentences."
)

prompt = template.invoke({
    "topic": "machine learning",
    "audience": "10-year-old",
    "sentences": 3
})
print(prompt.text)
print("\n---")
print(llm.invoke(prompt).content)

## 2 — Role Prompting

Tell the model WHO it is. This changes tone, depth, and format.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

role_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a friendly science teacher who uses simple analogies."),
    ("human", "Explain how a vector database works.")
])

print(llm.invoke(role_prompt.invoke({})).content)

## 3 — Few-Shot Prompting

Give examples of the desired input/output pattern.

In [ ]:
few_shot_prompt = ChatPromptTemplate.from_messages([
    ("system", "Classify the sentiment as Positive, Neutral, or Negative."),
    ("human", "The product arrived early and works perfectly."),
    ("ai", "Positive"),
    ("human", "It is okay, nothing special."),
    ("ai", "Neutral"),
    ("human", "The app crashes every time I open it."),
    ("ai", "Negative"),
    ("human", "{text}")
])

chain = few_shot_prompt | llm
print(chain.invoke({"text": "Delivery was late but the quality is excellent."}).content)

## 4 — Output Format Instructions

Ask the model to return JSON, bullet points, or tables.

In [ ]:
format_prompt = ChatPromptTemplate.from_messages([
    ("system", "You return only valid JSON with keys: name, age, skills (list)."),
    ("human", "Create a profile for a senior Python AI engineer.")
])

print(llm.invoke(format_prompt.invoke({})).content)

## 5 — Pydantic Structured Output

Force the model to return a typed Python object you can trust.

In [ ]:
from pydantic import BaseModel, Field
from typing import List

class Profile(BaseModel):
    name: str = Field(description="Person's name")
    age: int = Field(description="Person's age")
    skills: List[str] = Field(description="List of professional skills")
    summary: str = Field(description="One-line bio")

structured_llm = llm.with_structured_output(Profile)

profile_prompt = ChatPromptTemplate.from_messages([
    ("system", "Create a realistic profile for the requested role."),
    ("human", "{role}")
])

profile_chain = profile_prompt | structured_llm
profile = profile_chain.invoke({"role": "MLOps engineer with 5 years experience"})
print(profile)
print(type(profile))

## Exercises

1. Build a prompt that rewrites user text in three tones: formal, casual, professional.
2. Add 3 more examples to the few-shot sentiment classifier.
3. Create a Pydantic model for a `Book` and generate 5 book recommendations for a genre.
4. Compare outputs at temperature 0.0, 0.5, and 1.0 for the same prompt.